# Random Forest Model for Laptop Price Prediction

This notebook trains a Random Forest model with hyperparameter tuning, evaluates performance metrics, and compares with other models.

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from pydantic import BaseModel, Field
import joblib
import os

# Load preprocessed data
train_df_encoded = pd.read_csv('../../data/processed/training/train_data_v0.csv')
val_df_encoded = pd.read_csv('../../data/processed/training/val_data_v0.csv')
test_df_encoded = pd.read_csv('../../data/processed/training/test_data_v0.csv')

print(f"Train set: {train_df_encoded.shape}")
print(f"Validation set: {val_df_encoded.shape}")
print(f"Test set: {test_df_encoded.shape}")

Train set: (6851, 22)
Validation set: (762, 22)
Test set: (846, 22)


In [4]:
# ==================== DATA PREPARATION ====================
X_train = train_df_encoded.drop('Price', axis=1).values
y_train = train_df_encoded['Price'].values
X_val = val_df_encoded.drop('Price', axis=1).values
y_val = val_df_encoded['Price'].values
X_test = test_df_encoded.drop('Price', axis=1).values
y_test = test_df_encoded['Price'].values

print(f"Training set shape: {X_train.shape}")
print(f"Validation set shape: {X_val.shape}")
print(f"Test set shape: {X_test.shape}")

# ==================== TRAIN RANDOM FOREST MODEL ====================
# Grid search for best hyperparameters
print("\n" + "="*80)
print("GRID SEARCH: Finding optimal Random Forest hyperparameters")
print("="*80)

best_val_mse = float('inf')
best_params = {}
best_model = None

# Grid search parameters
n_estimators_list = [50, 100, 150, 200]
max_depths = [10, 15, 20, None]

for n_est in n_estimators_list:
    for max_d in max_depths:
        rf = RandomForestRegressor(
            n_estimators=n_est,
            max_depth=max_d,
            min_samples_split=5,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1
        )
        rf.fit(X_train, y_train)
        
        y_val_pred = rf.predict(X_val)
        val_mse = mean_squared_error(y_val, y_val_pred)
        val_mae = mean_absolute_error(y_val, y_val_pred)
        val_mape = np.mean(np.abs((y_val - y_val_pred) / y_val)) * 100
        
        print(f"n_est={n_est:3d}, max_d={str(max_d):4s}: MSE={val_mse:12,.0f} | MAE=${val_mae:8.2f} | MAPE={val_mape:6.2f}%")
        
        if val_mse < best_val_mse:
            best_val_mse = val_mse
            best_params = {'n_estimators': n_est, 'max_depth': max_d}
            best_model = rf

print(f"\n{'▼'*80}")
print(f"BEST PARAMS: n_estimators={best_params['n_estimators']}, max_depth={best_params['max_depth']}")
print(f"Best Validation MSE: {best_val_mse:,.2f}")
print(f"{'▼'*80}")

# Use best model
rf_model = best_model

# ==================== EVALUATE MODEL ====================
print("\n" + "="*80)
print("MODEL EVALUATION ON ALL DATASETS")
print("="*80)

# Training set evaluation
y_train_pred = rf_model.predict(X_train)
train_mse = mean_squared_error(y_train, y_train_pred)
train_rmse = np.sqrt(train_mse)
train_mae = mean_absolute_error(y_train, y_train_pred)
train_mape = np.mean(np.abs((y_train - y_train_pred) / y_train)) * 100
train_r2 = r2_score(y_train, y_train_pred)

print(f"\n{'TRAINING SET':-^80}")
print(f"  • MSE:    {train_mse:>15,.2f}")
print(f"  • RMSE:   ${train_rmse:>14.2f}  (Average prediction error in dollars)")
print(f"  • MAE:    ${train_mae:>14.2f}  (Mean Absolute Error)")
print(f"  • MAPE:   {train_mape:>14.2f}%  (Mean Absolute Percentage Error)")
print(f"  • R²:     {train_r2:>15.4f}  (Explains {train_r2*100:.1f}% of price variance)")

# Validation set evaluation
y_val_pred = rf_model.predict(X_val)
val_mse = mean_squared_error(y_val, y_val_pred)
val_rmse = np.sqrt(val_mse)
val_mae = mean_absolute_error(y_val, y_val_pred)
val_mape = np.mean(np.abs((y_val - y_val_pred) / y_val)) * 100
val_r2 = r2_score(y_val, y_val_pred)

print(f"\n{'VALIDATION SET':-^80}")
print(f"  • MSE:    {val_mse:>15,.2f}")
print(f"  • RMSE:   ${val_rmse:>14.2f}  (Average prediction error in dollars)")
print(f"  • MAE:    ${val_mae:>14.2f}  (Mean Absolute Error)")
print(f"  • MAPE:   {val_mape:>14.2f}%  (Mean Absolute Percentage Error)")
print(f"  • R²:     {val_r2:>15.4f}  (Explains {val_r2*100:.1f}% of price variance)")

# Test set evaluation
y_test_pred = rf_model.predict(X_test)
test_mse = mean_squared_error(y_test, y_test_pred)
test_rmse = np.sqrt(test_mse)
test_mae = mean_absolute_error(y_test, y_test_pred)
test_mape = np.mean(np.abs((y_test - y_test_pred) / y_test)) * 100
test_r2 = r2_score(y_test, y_test_pred)

print(f"\n{'TEST SET':-^80}")
print(f"  • MSE:    {test_mse:>15,.2f}")
print(f"  • RMSE:   ${test_rmse:>14.2f}  (Average prediction error in dollars)")
print(f"  • MAE:    ${test_mae:>14.2f}  (Mean Absolute Error)")
print(f"  • MAPE:   {test_mape:>14.2f}%  (Mean Absolute Percentage Error)")
print(f"  • R²:     {test_r2:>15.4f}  (Explains {test_r2*100:.1f}% of price variance)")

print("\n✓ Model training complete!")

# Save model
joblib.dump(rf_model, 'rf_model.pkl')
print("✓ Model saved to rf_model.pkl")

# Store feature names
feature_names = list(train_df_encoded.drop('Price', axis=1).columns)
print(f"\nFeature names ({len(feature_names)} total): {feature_names}")

Training set shape: (6851, 21)
Validation set shape: (762, 21)
Test set shape: (846, 21)

GRID SEARCH: Finding optimal Random Forest hyperparameters
n_est= 50, max_d=10  : MSE=      78,873 | MAE=$  200.95 | MAPE= 21.37%
n_est= 50, max_d=15  : MSE=      70,112 | MAE=$  179.58 | MAPE= 19.28%
n_est= 50, max_d=20  : MSE=      68,908 | MAE=$  176.72 | MAPE= 18.94%
n_est= 50, max_d=None: MSE=      69,041 | MAE=$  176.80 | MAPE= 18.94%
n_est=100, max_d=10  : MSE=      79,445 | MAE=$  201.20 | MAPE= 21.44%
n_est=100, max_d=15  : MSE=      69,613 | MAE=$  179.12 | MAPE= 19.21%
n_est=100, max_d=20  : MSE=      68,693 | MAE=$  176.30 | MAPE= 18.88%
n_est=100, max_d=None: MSE=      68,880 | MAE=$  176.19 | MAPE= 18.86%
n_est=150, max_d=10  : MSE=      79,538 | MAE=$  200.78 | MAPE= 21.45%
n_est=150, max_d=15  : MSE=      69,762 | MAE=$  178.90 | MAPE= 19.21%
n_est=150, max_d=20  : MSE=      69,070 | MAE=$  176.34 | MAPE= 18.91%
n_est=150, max_d=None: MSE=      69,221 | MAE=$  176.17 | MAPE= 18.89%

In [5]:
# ==================== INFERENCE FUNCTIONS ====================

def infer_price_from_encoded_data_rf(encoded_data: pd.DataFrame) -> float:
    """Predict laptop price from already encoded data using Random Forest"""
    X = encoded_data.values
    price = rf_model.predict(X)[0]
    return price

def infer_price_from_raw_input_rf(laptop_spec) -> dict:
    """Predict laptop price from raw human input using Random Forest"""
    # Get CPU and GPU marks (placeholder - to be implemented)
    cpu_mark = 12675.0
    gpu_mark = 2000.0
    
    # Parse resolution
    width, height = map(int, laptop_spec.resolution.split('x'))
    
    # Parse RAM and Storage
    if laptop_spec.ram.upper().endswith('TB'):
        ram_gb = float(laptop_spec.ram[:-2]) * 1024
    else:
        ram_gb = float(laptop_spec.ram.replace('GB', '').strip())
    
    if laptop_spec.storage.upper().endswith('TB'):
        storage_gb = float(laptop_spec.storage[:-2]) * 1024
    else:
        storage_gb = float(laptop_spec.storage.replace('GB', '').strip())
    
    # Create feature dictionary
    data_dict = {}
    data_dict['CPU Mark'] = cpu_mark
    data_dict['GPU Mark'] = gpu_mark
    data_dict['Monitor'] = float(laptop_spec.monitor)
    data_dict['Width'] = width
    data_dict['Height'] = height
    data_dict['RAM'] = ram_gb
    data_dict['Storage Amount'] = storage_gb
    data_dict['Weight'] = float(laptop_spec.weight)
    
    # Initialize all categorical features to 0
    known_brands = ['Brand_Acer', 'Brand_Apple', 'Brand_Asus', 'Brand_Dell', 'Brand_HP', 
                    'Brand_LG', 'Brand_Lenovo', 'Brand_MSI', 'Brand_Microsoft', 'Brand_Other']
    known_os = ['OS_Other', 'OS_Windows 10', 'OS_Windows 11']
    
    for brand_col in known_brands:
        data_dict[brand_col] = 0
    for os_col in known_os:
        data_dict[os_col] = 0
    
    # Set brand
    brand_clean = laptop_spec.brand.lower()
    brand_mapping = {
        'acer': 'Brand_Acer', 'apple': 'Brand_Apple', 'asus': 'Brand_Asus',
        'dell': 'Brand_Dell', 'hp': 'Brand_HP', 'lg': 'Brand_LG',
        'lenovo': 'Brand_Lenovo', 'msi': 'Brand_MSI', 'microsoft': 'Brand_Microsoft'
    }
    
    if brand_clean in brand_mapping:
        data_dict[brand_mapping[brand_clean]] = 1
    else:
        data_dict['Brand_Other'] = 1
    
    # Set OS
    os_clean = laptop_spec.os.lower()
    if 'windows 11' in os_clean:
        data_dict['OS_Windows 11'] = 1
    elif 'windows 10' in os_clean:
        data_dict['OS_Windows 10'] = 1
    else:
        data_dict['OS_Other'] = 1
    
    # Create DataFrame
    X = pd.DataFrame([data_dict])[feature_names]
    predicted_price = rf_model.predict(X.values)[0]
    
    return {
        'predicted_price': float(predicted_price),
        'specification': laptop_spec.model_dump(),
        'cpu_mark': cpu_mark,
        'gpu_mark': gpu_mark
    }


# ==================== TEST INFERENCE ====================
print("\n" + "="*80)
print("INFERENCE TEST: Using encoded data from test set")
print("="*80)
sample_encoded = test_df_encoded.drop('Price', axis=1).iloc[0:1]
actual_price = test_df_encoded['Price'].iloc[0]
predicted_price_rf = infer_price_from_encoded_data_rf(sample_encoded)
error = abs(predicted_price_rf - actual_price)
error_pct = (error / actual_price) * 100

print(f"Actual Price:    ${actual_price:.2f}")
print(f"Predicted Price: ${predicted_price_rf:.2f}")
print(f"Error:           ${error:.2f} ({error_pct:.1f}%)")
print("="*80)


INFERENCE TEST: Using encoded data from test set
Actual Price:    $1599.00
Predicted Price: $1620.03
Error:           $21.03 (1.3%)
